In [2]:
# ================================
# 1️⃣ Install Dependencies
# ================================
!pip install -q transformers datasets scikit-learn tqdm

# ================================
# 2️⃣ Imports
# ================================
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

# Mixed precision
from torch.cuda.amp import autocast, GradScaler

# ================================
# 3️⃣ Device
# ================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ================================
# 4️⃣ Load Dataset
# ================================
file_path = "/content/CN_relations_fixed.json"
with open(file_path, "r") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ================================
# 5️⃣ Labels
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def encode_labels(example):
    example["label"] = label2id[example["output"].strip()]
    return example

dataset = dataset.map(encode_labels)

# ================================
# 6️⃣ Few-Shot Sampling
# ================================
# Select N examples per label
N_SHOT = 10
indices = []
for label_id in range(len(LABELS)):
    label_indices = [i for i, ex in enumerate(dataset) if ex["label"] == label_id]
    indices.extend(label_indices[:N_SHOT])

few_shot_dataset = Subset(dataset, indices)

# Split 80/20 for train/val
train_size = int(0.8 * len(few_shot_dataset))
train_dataset = Subset(few_shot_dataset, range(train_size))
val_dataset = Subset(few_shot_dataset, range(train_size, len(few_shot_dataset)))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, collate_fn=collate_fn)

# ================================
# 7️⃣ Tokenizer
# ================================
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def collate_fn(batch):
    texts = [ex["input"] for ex in batch]
    labels = torch.tensor([ex["label"] for ex in batch])
    encoding = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )
    encoding["labels"] = labels
    return encoding

# ================================
# 8️⃣ Model
# ================================
class DistilBERTClassifier(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.bert = base_model
        self.classifier = nn.Linear(base_model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

base_model = AutoModel.from_pretrained(model_name)
model = DistilBERTClassifier(base_model, len(LABELS)).to(device)

# ================================
# 9️⃣ Optimizer & Scheduler
# ================================
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()

# ================================
# 🔟 Training Loop (Few-Shot)
# ================================
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(train_loader):.4f}")

# ================================
# 1️⃣1️⃣ Evaluation
# ================================
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== FEW-SHOT RESULTS (DistilBERT) =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ================================
# 1️⃣2️⃣ Save Predictions
# ================================
results = [{"input": ex["input"], "ground_truth": ex["output"].strip(), "predicted": id2label[p]} for ex, p in zip(val_dataset, y_pred)]

output_file = "/content/distilbert_fewshot_predictions.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"✅ Saved predictions to {output_file}")

from google.colab import files
files.download(output_file)

Using device: cuda


Map:   0%|          | 0/1952 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3603/3783674118.py:119: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/13 [00:00<?, ?it/s]/tmp/ipykernel_3603/3783674118.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_3603/3783674118.py:140: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.st

Epoch 1 Avg Loss: 1.9325


...

Epoch 3 Avg Loss: 1.3393


100%|██████████| 4/4 [00:00<00:00, 30.05it/s]



===== FEW-SHOT RESULTS (DistilBERT) =====
Accuracy : 0.0769
Precision: 0.0769
Recall   : 0.0769
F1 Score : 0.0769
✅ Saved predictions to /content/distilbert_fewshot_predictions.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>